# Week 4 — Final Evaluation Suite Using the Official Team Question Bank

**Project:** BC Federal Labor Jurisdiction Engine  
**Stack:** Flowise + Ollama + FAISS + `qwen2.5:3b`

This version uses the **exact 40 questions** from the uploaded team spreadsheet:

`Question Bank - Evaluation Spreadsheet.xlsx`

Question distribution:

- 15 Federal questions (`F‑01`–`F‑15`)
- 15 BC Provincial questions (`BC‑01`–`BC‑15`)
- 10 jurisdiction trick questions (`T‑01`–`T‑10`)

The notebook runs each question sequentially through the local Flowise Prediction API, saves a checkpoint after every question, and produces a manual scoring sheet for retrieval, generation, citation, and refusal evaluation.


In [ ]:
# Run once only if a package is unavailable:
# %pip install requests pandas

import json
import time
from pathlib import Path
from typing import Any

import pandas as pd
import requests

FLOWISE_SERVER_URL = "http://localhost:3000"

# Confirm this matches the current EVALUATION / FAISS chatflow URL.
CHATFLOW_ID = "197e3aa1-375d-4d7c-a9f4-0d5d14ae4199"

OLLAMA_URL = "http://localhost:11434"
FLOWISE_API_URL = f"{FLOWISE_SERVER_URL}/api/v1/prediction/{CHATFLOW_ID}"
HEADERS = {"Content-Type": "application/json"}

TIMEOUT_SECONDS = 300
MAX_RETRIES = 1
PAUSE_BETWEEN_QUESTIONS = 1
RESUME_EXISTING_RESULTS = True

# Leave as None to run all 40 questions.
# For a quick test, temporarily use 3.
RUN_LIMIT = None

OUTPUT_DIR = Path("week4_outputs_official_bank")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_JSON = OUTPUT_DIR / "evaluation_run_results.json"
RESULTS_CSV = OUTPUT_DIR / "evaluation_run_results.csv"
SCORECARD_CSV = OUTPUT_DIR / "evaluation_scorecard.csv"
SUMMARY_JSON = OUTPUT_DIR / "evaluation_summary.json"
ERROR_CASES_CSV = OUTPUT_DIR / "evaluation_error_cases.csv"

REFUSAL_TEXT = (
    "I cannot find a verified parameter for this specific jurisdiction "
    "layout within the official guides."
)


In [ ]:
def check_service(url: str, name: str) -> None:
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        print(f"✅ {name} is reachable: {url}")
    except requests.RequestException as exc:
        raise RuntimeError(
            f"{name} is not reachable at {url}. "
            f"Start the service and rerun this cell. Original error: {exc}"
        ) from exc

check_service(f"{FLOWISE_SERVER_URL}/api/v1/ping", "Flowise")
check_service(f"{OLLAMA_URL}/api/tags", "Ollama")

models_response = requests.get(f"{OLLAMA_URL}/api/tags", timeout=10)
models_response.raise_for_status()

installed_models = {
    item.get("name", "")
    for item in models_response.json().get("models", [])
}

print("\nInstalled Ollama models:")
for model_name in sorted(installed_models):
    print(f"  - {model_name}")

required_prefixes = ["nomic-embed-text", "qwen2.5:3b"]
missing_models = [
    required
    for required in required_prefixes
    if not any(name.startswith(required) for name in installed_models)
]

if missing_models:
    raise RuntimeError(
        "Missing required Ollama model(s): " + ", ".join(missing_models)
    )

print("✅ Required Ollama models are available.")


## Official 40-question bank

The `Verbatim Truth / Expected Rule` field is retained as the team's evaluation reference.  
For trick questions (`T‑01`–`T‑10`), the current chatbot prompt is expected to trigger the exact refusal response when the user asks to apply the wrong jurisdiction.

**Important:** `BC‑01` explicitly contains a 2024 example rate. Score it according to the benchmark/corpus version approved by the team, and document any corpus-version conflict in the notes column.


In [ ]:
question_bank = [{'id': 'F‑01', 'user_query': 'What is the minimum wage for a Scotiabank teller in Burnaby?', 'correct_jurisdiction': 'Federal', 'expected_rule': 'Banks are federally regulated; federal minimum wage applies.'}, {'id': 'F‑02', 'user_query': 'How many paid sick days does a Rogers call‑centre employee get?', 'correct_jurisdiction': 'Federal', 'expected_rule': 'CLC: Up to 10 paid sick days, accrued monthly after 30 days.'}, {'id': 'F‑03', 'user_query': 'Does an Air Canada flight attendant earn overtime after 8 hours?', 'correct_jurisdiction': 'Federal', 'expected_rule': 'CLC overtime: 1.5× after exceeding standard weekly hours.'}, {'id': 'F‑04', 'user_query': 'Can a Bell technician refuse unsafe work?', 'correct_jurisdiction': 'Federal', 'expected_rule': 'CLC Part II: Right to refuse dangerous work.'}, {'id': 'F‑05', 'user_query': 'How long is maternity leave for a WestJet employee?', 'correct_jurisdiction': 'Federal', 'expected_rule': '17 weeks maternity + up to 63 weeks parental.'}, {'id': 'F‑06', 'user_query': 'Does a TD Bank employee get holiday pay for working on Canada Day?', 'correct_jurisdiction': 'Federal', 'expected_rule': 'CLC: Paid statutory holiday rules apply.'}, {'id': 'F‑07', 'user_query': 'Can a CN Rail worker file a break complaint with BC ESA?', 'correct_jurisdiction': 'Federal', 'expected_rule': 'Interprovincial rail is federal; complaints go to Labour Program.'}, {'id': 'F‑08', 'user_query': 'What is the overtime rate for a FedEx interprovincial driver?', 'correct_jurisdiction': 'Federal', 'expected_rule': 'CLC: 1.5× overtime after standard weekly limits.'}, {'id': 'F‑09', 'user_query': 'Does a Vancouver‑based airline mechanic qualify for federal bereavement leave?', 'correct_jurisdiction': 'Federal', 'expected_rule': 'CLC: Up to 10 days, first 3 paid after 3 months.'}, {'id': 'F‑10', 'user_query': 'Can a bank teller use BC ESA rules for vacation pay?', 'correct_jurisdiction': 'Federal', 'expected_rule': 'Banks follow CLC vacation rules (2–3 weeks based on tenure).'}, {'id': 'F‑11', 'user_query': 'How many hours can a long‑haul truck driver work before overtime?', 'correct_jurisdiction': 'Federal', 'expected_rule': 'Federally regulated transport; CLC overtime rules apply.'}, {'id': 'F‑12', 'user_query': 'Does a postal worker get federal family violence leave?', 'correct_jurisdiction': 'Federal', 'expected_rule': 'CLC: Up to 10 days, first 5 paid.'}, {'id': 'F‑13', 'user_query': 'Can a telecom worker at Bell request federal compassionate care leave?', 'correct_jurisdiction': 'Federal', 'expected_rule': 'CLC: Up to 28 weeks unpaid compassionate care leave.'}, {'id': 'F‑14', 'user_query': 'What rest period must an Air Canada pilot receive between shifts?', 'correct_jurisdiction': 'Federal', 'expected_rule': 'Aviation is federal; CLC + Transport Canada duty‑time rules apply.'}, {'id': 'F‑15', 'user_query': 'Does a bank employee qualify for federal personal leave?', 'correct_jurisdiction': 'Federal', 'expected_rule': 'CLC: 5 days personal leave, first 3 paid after 3 months.'}, {'id': 'BC‑01', 'user_query': 'What is the minimum wage for a retail cashier in Vancouver?', 'correct_jurisdiction': 'BC Provincial', 'expected_rule': 'BC minimum wage applies (e.g., $17.40/hr in 2024).'}, {'id': 'BC‑02', 'user_query': 'How many paid sick days does a restaurant server get in BC?', 'correct_jurisdiction': 'BC Provincial', 'expected_rule': 'BC ESA: 5 paid sick days after 90 days.'}, {'id': 'BC‑03', 'user_query': 'Does a Vancouver tech worker earn overtime after 8 hours?', 'correct_jurisdiction': 'BC Provincial', 'expected_rule': '1.5× after 8 hrs/day; 2× after 12 hrs/day.'}, {'id': 'BC‑04', 'user_query': 'How much vacation pay does a retail worker get after 1 year?', 'correct_jurisdiction': 'BC Provincial', 'expected_rule': '4% vacation pay (2 weeks).'}, {'id': 'BC‑05', 'user_query': 'Can a Burnaby office worker refuse unsafe work?', 'correct_jurisdiction': 'BC Provincial', 'expected_rule': 'WorkSafeBC: Right to refuse unsafe work.'}, {'id': 'BC‑06', 'user_query': 'Is a part‑time barista entitled to statutory holiday pay?', 'correct_jurisdiction': 'BC Provincial', 'expected_rule': 'Must work 15 of previous 30 days.'}, {'id': 'BC‑07', 'user_query': 'Does a Vancouver restaurant cook get a 30‑minute meal break?', 'correct_jurisdiction': 'BC Provincial', 'expected_rule': '30‑minute unpaid break after 5 hours.'}, {'id': 'BC‑08', 'user_query': 'Can a retail worker be scheduled for 12 hours straight?', 'correct_jurisdiction': 'BC Provincial', 'expected_rule': 'Allowed; overtime applies after 8 hours.'}, {'id': 'BC‑09', 'user_query': 'How much notice must an employer give before terminating a cashier?', 'correct_jurisdiction': 'BC Provincial', 'expected_rule': '1 week after 3 months; 2 weeks after 12 months, etc.'}, {'id': 'BC‑10', 'user_query': 'Does a Vancouver office assistant qualify for family responsibility leave?', 'correct_jurisdiction': 'BC Provincial', 'expected_rule': 'Up to 5 unpaid days per year.'}, {'id': 'BC‑11', 'user_query': 'Does a restaurant worker earn double‑time on statutory holidays?', 'correct_jurisdiction': 'BC Provincial', 'expected_rule': 'If worked, paid 1.5× plus average day’s pay if eligible.'}, {'id': 'BC‑12', 'user_query': 'Can a tech worker be forced to work split shifts?', 'correct_jurisdiction': 'BC Provincial', 'expected_rule': 'Allowed; must have minimum 8 hours between shifts.'}, {'id': 'BC‑13', 'user_query': 'Does a cashier get paid for reporting to work but being sent home?', 'correct_jurisdiction': 'BC Provincial', 'expected_rule': 'Minimum 2 hours reporting pay.'}, {'id': 'BC‑14', 'user_query': 'Can a server be paid a lower wage because they earn tips?', 'correct_jurisdiction': 'BC Provincial', 'expected_rule': 'No tip‑differential wage in BC; full minimum wage applies.'}, {'id': 'BC‑15', 'user_query': 'Does a part‑time retail worker earn vacation pay?', 'correct_jurisdiction': 'BC Provincial', 'expected_rule': 'Yes; 4% minimum regardless of hours.'}, {'id': 'T‑01', 'user_query': 'Can a Vancouver city bus driver use BC ESA to file a missing‑pay complaint?', 'correct_jurisdiction': 'Federal', 'expected_rule': 'Transit (Coast Mountain Bus Company) is federally regulated.'}, {'id': 'T‑02', 'user_query': 'Can a WestJet pilot use BC ESA rules for overtime?', 'correct_jurisdiction': 'Federal', 'expected_rule': 'Aviation is federal; BC ESA does not apply.'}, {'id': 'T‑03', 'user_query': 'Can a Rogers employee in Burnaby claim BC sick‑leave rights?', 'correct_jurisdiction': 'Federal', 'expected_rule': 'Telecom is federal; BC ESA does not apply.'}, {'id': 'T‑04', 'user_query': 'Can a Vancouver restaurant worker use the Canada Labour Code for vacation pay?', 'correct_jurisdiction': 'BC Provincial', 'expected_rule': 'Restaurants are provincial.'}, {'id': 'T‑05', 'user_query': 'Does a CN Rail worker qualify for BC overtime rules?', 'correct_jurisdiction': 'Federal', 'expected_rule': 'Interprovincial rail is federal.'}, {'id': 'T‑06', 'user_query': 'Can a bank teller file a complaint with the BC Employment Standards Branch?', 'correct_jurisdiction': 'Federal', 'expected_rule': 'Banks are federally regulated.'}, {'id': 'T‑07', 'user_query': 'Can a Vancouver tech worker use federal maternity leave rules?', 'correct_jurisdiction': 'BC Provincial', 'expected_rule': 'Tech companies are provincial unless federally regulated.'}, {'id': 'T‑08', 'user_query': 'Does a Burnaby office worker get 10 federal sick days?', 'correct_jurisdiction': 'BC Provincial', 'expected_rule': 'BC ESA applies; federal sick‑day rules do not.'}, {'id': 'T‑09', 'user_query': 'Can a restaurant server use federal holiday pay rules?', 'correct_jurisdiction': 'BC Provincial', 'expected_rule': 'Restaurants fall under BC ESA.'}, {'id': 'T‑10', 'user_query': 'Can a Vancouver airport baggage handler use BC ESA?', 'correct_jurisdiction': 'Federal', 'expected_rule': 'Airport operations are federally regulated.'}]

question_bank_df = pd.DataFrame(question_bank)

question_bank_df["question_type"] = question_bank_df["id"].apply(
    lambda value: (
        "Trick / Refusal"
        if str(value).startswith("T")
        else "Standard Answer"
    )
)

display(question_bank_df)

assert len(question_bank) == 40, "Expected exactly 40 benchmark questions."
assert len({item["id"] for item in question_bank}) == 40, (
    "Question IDs must be unique."
)

print("✅ Official benchmark bank validated: 40 unique questions.")
print(question_bank_df["correct_jurisdiction"].value_counts())


In [ ]:
def extract_answer(response_data: dict[str, Any]) -> str:
    for key in ("text", "answer", "result", "output"):
        value = response_data.get(key)
        if isinstance(value, str) and value.strip():
            return value.strip()

    for parent_key in ("data", "result", "output"):
        nested = response_data.get(parent_key)
        if isinstance(nested, dict):
            for key in ("text", "answer", "result", "output"):
                value = nested.get(key)
                if isinstance(value, str) and value.strip():
                    return value.strip()

    return ""

def extract_source_documents(
    response_data: dict[str, Any]
) -> list[dict[str, Any]]:
    for key in ("sourceDocuments", "source_documents", "documents", "sources"):
        value = response_data.get(key)
        if isinstance(value, list):
            return [item for item in value if isinstance(item, dict)]

    for parent_key in ("data", "result", "output"):
        nested = response_data.get(parent_key)
        if isinstance(nested, dict):
            for key in (
                "sourceDocuments",
                "source_documents",
                "documents",
                "sources",
            ):
                value = nested.get(key)
                if isinstance(value, list):
                    return [item for item in value if isinstance(item, dict)]

    return []

def get_source_origin(document: dict[str, Any]) -> str:
    metadata = document.get("metadata", {})
    blob = metadata.get("blob", {})

    return str(
        metadata.get("source")
        or metadata.get("filePath")
        or metadata.get("filename")
        or (blob.get("name") if isinstance(blob, dict) else "")
        or "Local Ingest Block"
    )

def classify_source(source: str) -> str:
    normalized = source.lower().replace("\\", "/")

    if "/federal/" in normalized or "canada labour code" in normalized:
        return "Federal"

    if "/bc/" in normalized or "british columbia" in normalized:
        return "BC Provincial"

    if "/union/" in normalized or "bcgeu" in normalized:
        return "Union"

    return "Unknown"

def deduplicate(values: list[str]) -> list[str]:
    seen = set()
    output = []

    for value in values:
        if value not in seen:
            seen.add(value)
            output.append(value)

    return output

def automatic_diagnostics(
    item: dict[str, Any],
    answer: str,
    sources: list[str],
) -> dict[str, Any]:
    answer_lower = answer.lower()
    is_trick = str(item["id"]).startswith("T")
    exact_refusal = answer.strip() == REFUSAL_TEXT

    source_groups = deduplicate(
        [classify_source(source) for source in sources]
    )

    expected_group = item["correct_jurisdiction"]
    retrieval_suggestion = int(expected_group in source_groups)

    contains_source_label = "source:" in answer_lower
    contains_internal_or_placeholder_citation = any(
        phrase in answer_lower
        for phrase in (
            "section xx",
            "section xxx",
            "doc id",
            "document id",
            "chunk id",
        )
    )

    warning_flags = []

    if not sources:
        warning_flags.append("no_source_documents")

    if retrieval_suggestion == 0:
        warning_flags.append("expected_jurisdiction_not_in_sources")

    if is_trick and not exact_refusal:
        warning_flags.append("required_refusal_not_triggered")

    if not is_trick and exact_refusal:
        warning_flags.append("unexpected_refusal")

    if not is_trick and not contains_source_label:
        warning_flags.append("missing_source_label")

    if contains_internal_or_placeholder_citation:
        warning_flags.append("invalid_internal_or_placeholder_citation")

    if (
        ("bank" in item["user_query"].lower() or "rbc" in item["user_query"].lower())
        and "transportation company" in answer_lower
    ):
        warning_flags.append("bank_misclassified_as_transportation")

    return {
        "is_trick_question": is_trick,
        "source_groups": source_groups,
        "exact_refusal": exact_refusal,
        "retrieval_score_suggestion": retrieval_suggestion,
        "refusal_score_suggestion": (
            int(exact_refusal) if is_trick else "N/A"
        ),
        "contains_source_label": contains_source_label,
        "contains_internal_or_placeholder_citation": (
            contains_internal_or_placeholder_citation
        ),
        "warning_flags": warning_flags,
    }

def save_results(records: list[dict[str, Any]]) -> None:
    with RESULTS_JSON.open("w", encoding="utf-8") as file:
        json.dump(records, file, indent=2, ensure_ascii=False)

    pd.DataFrame(records).to_csv(
        RESULTS_CSV,
        index=False,
        encoding="utf-8-sig",
    )


## Run the official evaluation

The loop:

- runs sequentially;
- gives every question an independent `sessionId`;
- retries one time after a timeout;
- saves after every question;
- resumes completed successful IDs when rerun.

At about one minute per answer, all 40 questions may take roughly 40–60 minutes.


In [ ]:
questions_to_run = question_bank[:RUN_LIMIT] if RUN_LIMIT else question_bank

existing_records: list[dict[str, Any]] = []

if RESUME_EXISTING_RESULTS and RESULTS_JSON.exists():
    with RESULTS_JSON.open("r", encoding="utf-8") as file:
        loaded = json.load(file)

    if isinstance(loaded, list):
        existing_records = [
            record for record in loaded if isinstance(record, dict)
        ]

completed_ids = {
    str(record["id"])
    for record in existing_records
    if str(record.get("status", "")).lower() == "success"
}

evaluation_logs = existing_records.copy()

print("🚨 INITIALIZING OFFICIAL 40-QUESTION COMPLIANCE AUDIT 🚨")
print("=" * 76)
print(f"Endpoint: {FLOWISE_API_URL}")
print(f"Questions selected: {len(questions_to_run)}")
print(f"Previously completed IDs: {sorted(completed_ids)}")
print("=" * 76)

session = requests.Session()

for position, item in enumerate(questions_to_run, 1):
    question_id = str(item["id"])

    if question_id in completed_ids:
        print(f"⏭️ Skipping completed question {question_id}")
        continue

    query = item["user_query"]

    payload = {
        "question": query,
        "overrideConfig": {
            "returnSourceDocuments": True,
            "sessionId": (
                f"week4-{CHATFLOW_ID[:8]}-"
                f"{question_id.replace('‑', '-').replace('–', '-')}"
            ),
        },
    }

    print("\n" + "=" * 76)
    print(f"📋 QUESTION {position}/{len(questions_to_run)} — {question_id}")
    print(f"Correct jurisdiction : {item['correct_jurisdiction']}")
    print(f"Expected rule        : {item['expected_rule']}")
    print(f"User query           : {query}")

    started_at = time.perf_counter()
    final_record = None
    last_error = ""

    for attempt in range(1, MAX_RETRIES + 2):
        try:
            print(f"⏳ Attempt {attempt}/{MAX_RETRIES + 1}...")

            response = session.post(
                FLOWISE_API_URL,
                json=payload,
                headers=HEADERS,
                timeout=TIMEOUT_SECONDS,
            )

            elapsed_seconds = round(time.perf_counter() - started_at, 2)
            response.raise_for_status()
            response_data: dict[str, Any] = response.json()

            answer = extract_answer(response_data)
            source_documents = extract_source_documents(response_data)

            source_paths = deduplicate(
                [get_source_origin(doc) for doc in source_documents]
            )

            source_details = []
            for document in source_documents:
                metadata = document.get("metadata", {})
                content = (
                    document.get("pageContent")
                    or document.get("text")
                    or document.get("content")
                    or ""
                )

                source_details.append({
                    "source": get_source_origin(document),
                    "metadata": metadata,
                    "content_preview": str(content)[:500],
                })

            diagnostics = automatic_diagnostics(
                item=item,
                answer=answer,
                sources=source_paths,
            )

            print(f"✅ HTTP status: {response.status_code}")
            print(f"⏱️ Response time: {elapsed_seconds} seconds")

            print("\nAI Answer:")
            print(answer if answer else "(No answer returned.)")

            print("\nRetrieved Sources:")
            if source_paths:
                for source_number, source_path in enumerate(source_paths, 1):
                    print(f"  {source_number}. {source_path}")
            else:
                print("  No source documents returned.")

            if diagnostics["warning_flags"]:
                print("\n⚠️ Automatic warning flags:")
                for flag in diagnostics["warning_flags"]:
                    print(f"  - {flag}")

            final_record = {
                "id": question_id,
                "user_query": query,
                "correct_jurisdiction": item["correct_jurisdiction"],
                "expected_rule": item["expected_rule"],
                "status": "success",
                "http_status": response.status_code,
                "response_seconds": elapsed_seconds,
                "answer": answer,
                "sources": source_paths,
                "source_details": source_details,
                **diagnostics,
                "error": "",
            }
            break

        except requests.exceptions.ReadTimeout as exc:
            elapsed_seconds = round(time.perf_counter() - started_at, 2)
            last_error = f"ReadTimeout: {exc}"
            print(
                f"❌ Timeout on {question_id} "
                f"after {elapsed_seconds} seconds."
            )

        except requests.RequestException as exc:
            elapsed_seconds = round(time.perf_counter() - started_at, 2)
            last_error = f"RequestException: {exc}"
            print(f"❌ Request error on {question_id}: {exc}")

        except (ValueError, TypeError, json.JSONDecodeError) as exc:
            elapsed_seconds = round(time.perf_counter() - started_at, 2)
            last_error = f"ResponseParsingError: {exc}"
            print(f"❌ Response parsing error on {question_id}: {exc}")

        if attempt <= MAX_RETRIES:
            print("Waiting 5 seconds before retry...")
            time.sleep(5)

    if final_record is None:
        is_trick = question_id.startswith("T")

        final_record = {
            "id": question_id,
            "user_query": query,
            "correct_jurisdiction": item["correct_jurisdiction"],
            "expected_rule": item["expected_rule"],
            "status": "error",
            "http_status": "",
            "response_seconds": elapsed_seconds,
            "answer": "",
            "sources": [],
            "source_details": [],
            "is_trick_question": is_trick,
            "source_groups": [],
            "exact_refusal": False,
            "retrieval_score_suggestion": 0,
            "refusal_score_suggestion": 0 if is_trick else "N/A",
            "contains_source_label": False,
            "contains_internal_or_placeholder_citation": False,
            "warning_flags": ["request_failed"],
            "error": last_error,
        }

    evaluation_logs = [
        record
        for record in evaluation_logs
        if str(record.get("id")) != question_id
    ]
    evaluation_logs.append(final_record)

    order_lookup = {
        item["id"]: index
        for index, item in enumerate(question_bank)
    }
    evaluation_logs.sort(
        key=lambda record: order_lookup.get(str(record["id"]), 999)
    )

    save_results(evaluation_logs)
    print(f"💾 Checkpoint saved: {RESULTS_JSON.resolve()}")

    time.sleep(PAUSE_BETWEEN_QUESTIONS)

print("\n" + "=" * 76)
print("✅ OFFICIAL WEEK 4 EVALUATION RUN FINISHED")
print(f"JSON results: {RESULTS_JSON.resolve()}")
print(f"CSV results : {RESULTS_CSV.resolve()}")
print("=" * 76)


In [ ]:
results_df = pd.DataFrame(evaluation_logs)

display_columns = [
    "id",
    "correct_jurisdiction",
    "status",
    "response_seconds",
    "source_groups",
    "exact_refusal",
    "warning_flags",
]

display(results_df[display_columns])

print("\nRun status counts:")
print(results_df["status"].value_counts(dropna=False))

successful_times = pd.to_numeric(
    results_df.loc[
        results_df["status"] == "success",
        "response_seconds",
    ],
    errors="coerce",
).dropna()

if not successful_times.empty:
    print(
        f"Average successful response time: "
        f"{successful_times.mean():.2f} seconds"
    )


## Manual scoring sheet

Automatic flags are diagnostic only. Human reviewers must compare the answer and retrieved source chunks against the team's `Verbatim Truth / Expected Rule`.

Enter:

- `retrieval_hit_manual`: 1 or 0
- `generation_faithfulness_manual`: 1 or 0
- `citation_quality_manual`: 1 or 0
- `refusal_accuracy_manual`: 1, 0, or N/A
- `root_cause_category`
- `notes`

Suggested root-cause categories:

- Correct
- Retrieval Collision
- Wrong Jurisdiction Retrieval
- Generation Hallucination
- Citation Failure
- Refusal Failure
- Corpus Coverage Gap
- Timeout / Technical Failure
- Benchmark / Corpus Version Conflict


In [ ]:
def flatten_list(value: Any) -> str:
    if isinstance(value, list):
        return " | ".join(str(item) for item in value)
    return str(value) if value is not None else ""

scorecard_rows = []

for record in evaluation_logs:
    scorecard_rows.append({
        "id": record["id"],
        "user_query": record["user_query"],
        "correct_jurisdiction": record["correct_jurisdiction"],
        "verbatim_truth_expected_rule": record["expected_rule"],
        "is_trick_question": record["is_trick_question"],
        "status": record["status"],
        "response_seconds": record["response_seconds"],
        "answer": record["answer"],
        "retrieved_sources": flatten_list(record["sources"]),
        "retrieved_source_groups": flatten_list(
            record.get("source_groups", [])
        ),
        "automatic_warning_flags": flatten_list(
            record.get("warning_flags", [])
        ),
        "suggested_retrieval_score": record.get(
            "retrieval_score_suggestion", ""
        ),
        "suggested_refusal_score": record.get(
            "refusal_score_suggestion", ""
        ),
        "retrieval_hit_manual": "",
        "generation_faithfulness_manual": "",
        "citation_quality_manual": "",
        "refusal_accuracy_manual": (
            "" if record["is_trick_question"] else "N/A"
        ),
        "root_cause_category": "",
        "notes": "",
    })

scorecard_df = pd.DataFrame(scorecard_rows)
scorecard_df.to_csv(
    SCORECARD_CSV,
    index=False,
    encoding="utf-8-sig",
)

display(scorecard_df.head())
print(f"✅ Manual scorecard created: {SCORECARD_CSV.resolve()}")


## Final metrics after manual review

Open `evaluation_scorecard.csv` in Excel, complete the manual columns, save the file, and rerun the next two cells.


In [ ]:
completed_scorecard = pd.read_csv(
    SCORECARD_CSV,
    dtype=str,
).fillna("")

def numeric_scores(column_name: str) -> pd.Series:
    return pd.to_numeric(
        completed_scorecard[column_name],
        errors="coerce",
    )

def metric_summary(scores: pd.Series) -> tuple[Any, int, int]:
    valid = scores.dropna()

    if valid.empty:
        return None, 0, 0

    passed = int(valid.sum())
    total = int(valid.count())
    percent = round((passed / total) * 100, 2)

    return percent, passed, total

retrieval = numeric_scores("retrieval_hit_manual")
generation = numeric_scores("generation_faithfulness_manual")
citation = numeric_scores("citation_quality_manual")

trick_mask = (
    completed_scorecard["is_trick_question"]
    .str.lower()
    .eq("true")
)
refusal = pd.to_numeric(
    completed_scorecard.loc[
        trick_mask,
        "refusal_accuracy_manual",
    ],
    errors="coerce",
)

metrics = {}

for label, values in (
    ("Retrieval Accuracy", retrieval),
    ("Generation Faithfulness", generation),
    ("Citation Quality", citation),
    ("Refusal Accuracy", refusal),
):
    percent, passed, total = metric_summary(values)
    metrics[label] = {
        "percent": percent,
        "passed": passed,
        "scored": total,
    }

summary_table = pd.DataFrame([
    {
        "Metric": metric,
        "Passed": values["passed"],
        "Scored": values["scored"],
        "Percent": values["percent"],
    }
    for metric, values in metrics.items()
])

display(summary_table)

summary_payload = {
    "total_questions": len(completed_scorecard),
    "metrics": metrics,
}

with SUMMARY_JSON.open("w", encoding="utf-8") as file:
    json.dump(summary_payload, file, indent=2, ensure_ascii=False)

print(f"✅ Summary saved: {SUMMARY_JSON.resolve()}")

if any(values["scored"] == 0 for values in metrics.values()):
    print(
        "\n⚠️ Some manual score columns are still blank. "
        "Complete the CSV in Excel and rerun this cell."
    )


In [ ]:
retrieval_values = numeric_scores("retrieval_hit_manual")
generation_values = numeric_scores("generation_faithfulness_manual")
citation_values = numeric_scores("citation_quality_manual")
refusal_values = pd.to_numeric(
    completed_scorecard["refusal_accuracy_manual"],
    errors="coerce",
)

failure_mask = (
    (retrieval_values == 0)
    | (generation_values == 0)
    | (citation_values == 0)
    | (refusal_values == 0)
    | (completed_scorecard["status"].str.lower() != "success")
)

error_cases = completed_scorecard.loc[failure_mask].copy()
error_cases.to_csv(
    ERROR_CASES_CSV,
    index=False,
    encoding="utf-8-sig",
)

print(f"Error cases found: {len(error_cases)}")

display(error_cases[
    [
        "id",
        "user_query",
        "automatic_warning_flags",
        "root_cause_category",
        "notes",
    ]
])

root_causes = (
    completed_scorecard["root_cause_category"]
    .replace("", pd.NA)
    .dropna()
    .value_counts()
    .rename_axis("Root Cause")
    .reset_index(name="Count")
)

print("\nRoot-cause counts:")
display(root_causes)

print(f"✅ Error cases saved: {ERROR_CASES_CSV.resolve()}")


## Recommended Week 4 screenshots

1. **Official bank verification**
   - the displayed 40-question table;
   - `Official benchmark bank validated: 40 unique questions`.

2. **Bulk API execution**
   - question ID;
   - expected rule;
   - HTTP 200;
   - AI answer;
   - retrieved sources;
   - response time.

3. **Completed evaluation**
   - `OFFICIAL WEEK 4 EVALUATION RUN FINISHED`;
   - JSON and CSV output paths.

4. **Manual evaluation summary**
   - Retrieval Accuracy;
   - Generation Faithfulness;
   - Citation Quality;
   - Refusal Accuracy;
   - one or more error cases and root-cause categories.
